# TetraFT — Kaggle runs

**Attach datasets**
- `tetraft-code` — flat `.py` + this notebook (**refresh** after code changes)
- `tetraft-fineweb-edu-50m` — `train.jsonl`, `val.jsonl`
- **B only:** Session A full `checkpoint-final`
- **P only:** Session B `checkpoint-final` (weights-only OK)
- **S / U:** code + FineWeb only — **fresh start, no ckpt**
- **L (layer map):** code + FineWeb + B `checkpoint-final`

| Setting | Value |
|---------|--------|
| Accelerator | **GPU** |
| Internet | **ON** first run |

| SESSION | Preset / script | Resume | Gate |
|---------|-----------------|--------|------|
| **U** | `scout_kl_bundle_r345_5m` | **fresh** 1280 | **&lt;49.31** ← **current next** |
| **L** | `run_layer_map.py` | B ckpt | role table + suggest |
| **A** | `heal_kl_50m` | none → 6104 | mid ≲50–52 |
| **B** | `heal_kl_50m` | A full → 12207 | done ~34.38 |
| **P** | `polish_kl_5m` | B → 13487 | **FAIL** — stop |
| **S** | `scout_kl_5m` + α/T | fresh 1280 | null — lock 0.5/2.0 |

**Current next:** `SESSION="U"` (R3 pre_rms + R4 unit_absmean + R5 LoRA r=8).  
Logic in `run_smoke.py` / `run_layer_map.py` — notebook is glue only.


In [ ]:
# Qwen3.5 needs recent transformers (model_type qwen3_5).
# If KeyError qwen3_5:
# %pip install -U "git+https://github.com/huggingface/transformers.git"
%pip install -q -U transformers accelerate bitsandbytes sentencepiece

In [ ]:
import sys
from pathlib import Path

def find_file(name: str) -> Path:
    roots = [Path("/kaggle/input"), Path("/kaggle/working"), Path(".")]
    for root in roots:
        if not root.exists():
            continue
        direct = root / name
        if direct.is_file():
            return direct
        for p in root.rglob(name):
            if p.is_file():
                return p
    raise FileNotFoundError(name)

code_py = find_file("run_smoke.py")
code_root = code_py.parent
sys.path.insert(0, str(code_root))
print("code:", code_root)

train_path = find_file("train.jsonl")
val_path = find_file("val.jsonl")
print("train:", train_path)
print("val:", val_path)

import transformers
print("transformers", transformers.__version__)

from config import SMOKE_PRESETS
assert "heal_kl_50m" in SMOKE_PRESETS, "heal_kl_50m missing — refresh tetraft-code"
assert "scout_kl_5m" in SMOKE_PRESETS
assert "scout_kl_bundle_r345_5m" in SMOKE_PRESETS, (
    "scout_kl_bundle_r345_5m missing — refresh tetraft-code"
)
assert (code_root / "run_layer_map.py").is_file(), "run_layer_map.py missing — refresh tetraft-code"
print("presets ok:", sorted(SMOKE_PRESETS))

In [ ]:
from run_smoke import run_smoke
from run_layer_map import main as run_layer_map_main
import argparse
import shutil
from pathlib import Path

# =============================================================================
# SESSION: "U" bundle | "L" layer map | "A" | "B" | "P" | "S" (α/T historical)
# =============================================================================
SESSION = "U"  # <-- U | L | A | B | P | S

# SESSION=S only (historical; static α/T null — prefer U)
SCOUT_ALPHA = 0.3
SCOUT_TEMPERATURE = 2.0
SCOUT_TAG = "a03_t2"

# SESSION=L only: path to B checkpoint-final (or leave None to find_file)
LAYER_MAP_CHECKPOINT = None  # e.g. "/kaggle/input/.../checkpoint-final"
LAYER_MAP_FP_MASK_TOPK = 8
LAYER_MAP_SKIP_PPL = False  # True = weight-only (CPU-friendlier)

CLEAR_OUTPUT = True

if SESSION.upper() == "U":
    # Bundle R3+R4+R5 @ ~5.24M — fresh; gate < 49.31
    PRESET = "scout_kl_bundle_r345_5m"
    MAX_STEPS = None
    SAVE_OPTIMIZER = False
    RESUME = None
    SKIP_SHOCK = False
    SKIP_ORIG = False
    DISTILL_ALPHA = None
    DISTILL_TEMPERATURE = None
    PRE_RMS = None  # preset True
    NO_PRE_RMS = False
    WEIGHT_CALIB = None  # preset unit_absmean
    LORA_RANK = None  # preset 8
    LORA_ALPHA = None
    OUTPUT_DIR = "/kaggle/working/checkpoints_scout_kl_bundle_r345_5m"
    print("bundle R3+R4+R5: pre_rms + unit_absmean + lora_rank=8")
    print("gate: end PPL < 49.31 (scout_kl_5m); if PASS → leave-one-out before long KL")
elif SESSION.upper() == "L":
    # D0 layer map — no training
    ckpt = LAYER_MAP_CHECKPOINT or str(find_file("checkpoint-final"))
    out_lm = Path("/kaggle/working/layer_map_b")
    if CLEAR_OUTPUT and out_lm.exists():
        shutil.rmtree(out_lm)
        print("cleared", out_lm)
    out_lm.mkdir(parents=True, exist_ok=True)
    argv = [
        "--checkpoint", ckpt,
        "--preset", "heal_kl_50m",
        "--val-data", str(val_path),
        "--max-eval-batches", "20",
        "--fp-mask-topk", str(int(LAYER_MAP_FP_MASK_TOPK)),
        "--output-dir", str(out_lm),
    ]
    if LAYER_MAP_SKIP_PPL:
        argv.append("--skip-ppl")
    print("layer map checkpoint:", ckpt)
    print("argv:", argv)
    rc = run_layer_map_main(argv)
    print("layer_map exit", rc)
    summary = out_lm / "layer_map_summary.json"
    if summary.is_file():
        import json
        with summary.open() as f:
            s = json.load(f)
        print("suggestion:", s.get("suggestion"))
        print("ppl_student:", s.get("ppl_student"), "ppl_fp_mask:", s.get("ppl_fp_mask"))
    raise SystemExit(rc)
elif SESSION.upper() == "A":
    PRESET = "heal_kl_50m"
    MAX_STEPS = 6104
    SAVE_OPTIMIZER = True
    RESUME = None
    SKIP_SHOCK = False
    SKIP_ORIG = False
    DISTILL_ALPHA = None
    DISTILL_TEMPERATURE = None
    PRE_RMS = None
    NO_PRE_RMS = False
    WEIGHT_CALIB = None
    LORA_RANK = None
    LORA_ALPHA = None
    OUTPUT_DIR = "/kaggle/working/checkpoints_heal_kl_50m_A"
elif SESSION.upper() == "B":
    PRESET = "heal_kl_50m"
    MAX_STEPS = 12207
    SAVE_OPTIMIZER = False
    RESUME = str(find_file("checkpoint-final"))
    SKIP_SHOCK = True
    SKIP_ORIG = True
    DISTILL_ALPHA = None
    DISTILL_TEMPERATURE = None
    PRE_RMS = None
    NO_PRE_RMS = False
    WEIGHT_CALIB = None
    LORA_RANK = None
    LORA_ALPHA = None
    OUTPUT_DIR = "/kaggle/working/checkpoints_heal_kl_50m_B"
    print("resume from:", RESUME)
elif SESSION.upper() == "P":
    PRESET = "polish_kl_5m"
    MAX_STEPS = None
    SAVE_OPTIMIZER = False
    RESUME = str(find_file("checkpoint-final"))
    SKIP_SHOCK = True
    SKIP_ORIG = True
    DISTILL_ALPHA = None
    DISTILL_TEMPERATURE = None
    PRE_RMS = None
    NO_PRE_RMS = False
    WEIGHT_CALIB = None
    LORA_RANK = None
    LORA_ALPHA = None
    OUTPUT_DIR = "/kaggle/working/checkpoints_polish_kl_5m"
    print("polish resume from:", RESUME)
elif SESSION.upper() == "S":
    PRESET = "scout_kl_5m"
    MAX_STEPS = None
    SAVE_OPTIMIZER = False
    RESUME = None
    SKIP_SHOCK = False
    SKIP_ORIG = False
    DISTILL_ALPHA = float(SCOUT_ALPHA)
    DISTILL_TEMPERATURE = float(SCOUT_TEMPERATURE)
    PRE_RMS = None
    NO_PRE_RMS = False
    WEIGHT_CALIB = None
    LORA_RANK = None
    LORA_ALPHA = None
    OUTPUT_DIR = f"/kaggle/working/checkpoints_scout_kl_{SCOUT_TAG}"
    print(f"α/T scout: α={DISTILL_ALPHA} T={DISTILL_TEMPERATURE} tag={SCOUT_TAG}")
    print("note: static α/T null — prefer SESSION=U bundle")
else:
    raise ValueError("SESSION must be 'U', 'L', 'A', 'B', 'P', or 'S'")

out = Path(OUTPUT_DIR)
if CLEAR_OUTPUT and out.exists():
    shutil.rmtree(out)
    print("cleared", out)
out.mkdir(parents=True, exist_ok=True)

ns = argparse.Namespace(
    preset=PRESET,
    model_name=None,
    train_data=str(train_path),
    val_data=str(val_path),
    output_dir=OUTPUT_DIR,
    seq_length=None,
    batch_size=None,
    max_steps=MAX_STEPS,
    max_eval_batches=20,
    max_train_texts=None,
    max_val_texts=None,
    skip_train=False,
    skip_shock=SKIP_SHOCK,
    skip_orig=SKIP_ORIG,
    resume=RESUME,
    no_bf16=False,
    no_8bit_adam=False,
    quant_warmup_steps=None,
    warmup_steps=None,
    learning_rate=None,
    lr_scheduler=None,
    min_lr_ratio=None,
    schedule_max_steps=None,
    logging_steps=None,
    eval_steps=None,
    save_steps=None,
    save_optimizer=SAVE_OPTIMIZER,
    skip_linear_attn=None,
    no_skip_linear_attn=False,
    distill_alpha=DISTILL_ALPHA,
    distill_temperature=DISTILL_TEMPERATURE,
    quant_reg_beta=None,
    pre_rms=PRE_RMS,
    no_pre_rms=NO_PRE_RMS,
    weight_calib=WEIGHT_CALIB,
    lora_rank=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    seed=42,
    device_map="auto",
)
print(
    f"SESSION={SESSION} preset={PRESET} max_steps={MAX_STEPS} "
    f"α={DISTILL_ALPHA} T={DISTILL_TEMPERATURE} resume={RESUME} out={OUTPUT_DIR}"
)
print("note: KL loads frozen FP teacher (~2× VRAM)")
results = run_smoke(ns)
keys = [
    "preset", "ppl_original", "ppl_shock", "ppl_after_smoke",
    "loss_finite", "tokens_seen", "tokens_budget", "steps_ran",
    "resumed_step", "schedule_horizon_steps", "distill",
]
print({k: results[k] for k in keys if k in results})
if "inventory_summary" in results:
    inv = results["inventory_summary"]
    print("inventory", inv)
    if inv.get("n_eligible", 0) > 150:
        print("WARNING: eligible looks like all-Linear — GDN skip may be off")
ppl = results.get("ppl_after_smoke")
if ppl is not None:
    ref = results.get("ppl_original") or results.get("ppl_original_ref") or 17.67
    print(f"after/orig ≈ {ppl / ref:.3f} (ref orig {ref})")
    if SESSION.upper() == "A":
        print(f"Session A mid PPL={ppl:.2f} — go/no-go: continue B if ≲50–52 and falling")
    elif SESSION.upper() == "B":
        print(f"Session B final PPL={ppl:.2f} — bar CE heal_50m ~43.77; strong if ≲35")
    elif SESSION.upper() == "P":
        print(f"Polish final PPL={ppl:.2f} — gate < 34.38 (FAIL expected — stop polish)")
    elif SESSION.upper() == "U":
        gate = 49.31
        print(f"Bundle R345 final PPL={ppl:.2f} — gate < {gate}")
        if ppl < gate:
            print("PASS — leave-one-out @ 5M before long KL (no-pre-rms / weight-calib none / lora-rank 0)")
            print("Do NOT only polish old B; fresh long KL with winning DNA")
        else:
            print("NO PASS — try single-knob R5 or R3; see RESULTS.md §5.9")
    else:
        gate = 49.31
        print(f"α/T scout final PPL={ppl:.2f} — gate < {gate} (historical)")
        if ppl < gate:
            print(f"PASS — lock α={DISTILL_ALPHA} T={DISTILL_TEMPERATURE}")
        else:
            print("NO PASS — keep α=0.5 T=2; prefer SESSION=U")

### Bundle scout (SESSION=U) — current next

Preset `scout_kl_bundle_r345_5m`:
- **R3** `pre_rms=True`
- **R4** `weight_calib=unit_absmean`
- **R5** `lora_rank=8` α=8

Gate: end PPL **&lt; 49.31**. Attach **code + FineWeb only**. Refresh `tetraft-code`.

If PASS → leave-one-out (edit `NO_PRE_RMS` / `WEIGHT_CALIB` / `LORA_RANK` in a copy of U, or CLI).  
If FAIL → single-knob or D1 length; see `RESULTS.md` §5.9.

### Layer map (SESSION=L)

Attach B `checkpoint-final` + FineWeb. Set `LAYER_MAP_CHECKPOINT` if multiple ckpts.  
Artifacts under `/kaggle/working/layer_map_b/`.

### Frozen baselines

| Run | ≈ tokens | Val PPL |
|-----|---------:|--------:|
| Original FP | — | ~17.7 |
| scout_kl_5m | 5.2M | **~49.31** |
| CE heal_50m | 50M | ~43.77 |
| heal_kl_50m A+B | 50M | **~34.38** |
| polish_kl_5m | +5.2M | FAIL — stop |
| D0 layer map | — | flat error; FP-mask hurt; suggest D1 |

Record new PPL in `RESULTS.md`.
